# Sequence Preparation & Sliding Window Generation
### B.Tech Mini Project: Aircraft Engine Predictive Maintenance (NASA C-MAPSS FD001)

This notebook prepares 3D sequential temporal tensors (`samples`, `window_length`, `features`) for recurrent and convolutional deep learning models (e.g., LSTM, 1D-CNN) using sliding time windows.

#### Five-Stage Workflow:
1. **Stage 1 - Reproduce Preprocessing:** Self-contained ingestion, RUL target generation, 80/20 engine-wise split, variance-based feature pruning, and `StandardScaler` normalization.
2. **Stage 2 - Generate Training Windows:** Extract overlapping 30-cycle sliding windows ($W=30$, stride=1) independently for each training engine; map target RUL from each window's final cycle.
3. **Stage 3 - Generate Validation Windows:** Extract 30-cycle sliding windows independently for each validation engine without crossing engine boundaries.
4. **Stage 4 - Generate Test Windows:** Extract the final 30 observed cycles for each of the 100 test engines and align with ground-truth test RUL values.
5. **Stage 5 - Validation & Inspection:** Programmatic integrity assertions, dimension reporting, window sample inspection, and discussion of the test evaluation protocol.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("NumPy version  :", np.__version__)
print("Pandas version :", pd.__version__)
print("Libraries successfully imported.")

NumPy version  : 2.5.4
Pandas version : 3.1.0
Libraries successfully imported.


---
## Stage 1: Reproduce Preprocessing Pipeline

To keep this notebook self-contained across separate Jupyter kernel sessions, we re-execute the exact preprocessing pipeline established in `04_feature_preprocessing.ipynb`:
1. **Data Loading:** Ingest `train_FD001.txt`, `test_FD001.txt`, and `RUL_FD001.txt` with standard 26-column naming.
2. **Target Labeling:** Calculate piecewise linear capped RUL: $\text{RUL} = \min(125, \max(\text{cycle}) - \text{cycle})$.
3. **Engine-Wise Split:** Split unique engine units using `train_test_split(test_size=0.20, random_state=42)` (80 train engines, 20 val engines).
4. **Variance Filtering:** Identify zero-variance candidate features strictly on the training partition; retain near-constant `sensor_6`.
5. **Feature Scaling:** Fit `StandardScaler` strictly on the training partition; transform train, validation, and test features.

In [2]:
# 1. Define standard column schema
columns = (
    ["unit", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

# 2. Resolve dataset directory
DATA_DIR = Path("../data/raw/CMaps") if Path("../data/raw/CMaps").exists() else Path("data/raw/CMaps")

# 3. Ingest raw datasets
train_df = pd.read_csv(DATA_DIR / "train_FD001.txt", sep=r"\s+", header=None, names=columns)
test_df = pd.read_csv(DATA_DIR / "test_FD001.txt", sep=r"\s+", header=None, names=columns)
rul_df = pd.read_csv(DATA_DIR / "RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

# 4. Generate piecewise linear capped RUL labels
train_df["max_cycle"] = train_df.groupby("unit")["cycle"].transform("max")
train_df["raw_RUL"] = train_df["max_cycle"] - train_df["cycle"]
train_df["RUL"] = np.minimum(125, train_df["raw_RUL"])

# 5. Engine-wise split (80% train, 20% validation, fixed random_state=42)
unique_units = train_df["unit"].unique()
train_units, val_units = train_test_split(unique_units, test_size=0.20, random_state=42)

train_split_df = train_df[train_df["unit"].isin(train_units)].copy().reset_index(drop=True)
val_split_df = train_df[train_df["unit"].isin(val_units)].copy().reset_index(drop=True)

# 6. Dynamic feature selection strictly on training partition
candidate_features = [f"op_setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]
train_var = train_split_df[candidate_features].var()
train_nunique = train_split_df[candidate_features].nunique()

zero_var_features = [f for f in candidate_features if train_nunique[f] == 1 or train_var[f] < 1e-12]
selected_features = [f for f in candidate_features if f not in zero_var_features]

# 7. Fit StandardScaler strictly on training partition and transform all splits
scaler = StandardScaler()
train_split_df[selected_features] = scaler.fit_transform(train_split_df[selected_features])
val_split_df[selected_features] = scaler.transform(val_split_df[selected_features])
test_df[selected_features] = scaler.transform(test_df[selected_features])

# 8. Consistency Assertions matching Notebook 04
assert len(train_units) == 80 and len(val_units) == 20, "Split error!"
assert len(set(train_units).intersection(set(val_units))) == 0, "Overlap error!"
assert len(zero_var_features) == 7, "Unexpected zero-variance count!"
assert len(selected_features) == 17, "Unexpected selected feature count!"
assert "sensor_6" in selected_features, "sensor_6 was improperly removed!"

print("=" * 65)
print("STAGE 1: PREPROCESSING REPRODUCED & VERIFIED")
print("=" * 65)
print(f"Training Partition   : {len(train_units)} engines ({len(train_split_df):,} rows)")
print(f"Validation Partition : {len(val_units)} engines ({len(val_split_df):,} rows)")
print(f"Test Dataset         : {test_df['unit'].nunique()} engines ({len(test_df):,} rows)")
print(f"Removed Features (7) : {zero_var_features}")
print(f"Selected Features(17): {selected_features}")
print("=" * 65)

STAGE 1: PREPROCESSING REPRODUCED & VERIFIED
Training Partition   : 80 engines (16,561 rows)
Validation Partition : 20 engines (4,070 rows)
Test Dataset         : 100 engines (13,096 rows)
Removed Features (7) : ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Selected Features(17): ['op_setting_1', 'op_setting_2', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


---
## Stage 2: Generate Training Windows

### Sliding Window Formulation ($W=30$, Stride=1)
Sequential models (such as LSTMs and 1D-CNNs) take as input a continuous sequence of observations of fixed length $W=30$.

For each individual engine trajectory of length $L$:
1. Observations are sorted in temporal order by flight cycle: $[t_1, t_2, \dots, t_L]$.
2. A window of length $W=30$ cycles spans time indices $[k, k + W - 1]$.
3. The window slides forward by one cycle each time (stride $S=1$), producing exactly $L - W + 1$ overlapping temporal windows.
4. **Input Matrix ($X$):** The 2D slice of scaled selected feature values of shape $(30, 17)$.
5. **Target Label ($y$):** The ground-truth RUL value corresponding to the **final observation of the window** (cycle $k + W - 1$), representing the engine's remaining useful life at the moment of prediction.

In [3]:
WINDOW_SIZE = 30

def extract_sliding_windows(df, seq_len, feature_cols, target_col="RUL"):
    """
    Extracts 30-cycle sliding temporal windows independently per engine trajectory.
    
    Parameters:
        df (pd.DataFrame): Dataframe containing 'unit', 'cycle', features, and target.
        seq_len (int): Temporal window length (30).
        feature_cols (list): List of scaled feature column names.
        target_col (str): Target column name ('RUL').
        
    Returns:
        X (np.ndarray): 3D float32 array of shape (num_windows, seq_len, num_features)
        y (np.ndarray): 1D float32 array of shape (num_windows,)
        metadata (list of dict): Tracking info (unit, start_cycle, end_cycle, target)
    """
    X_list, y_list, metadata_list = [], [], []
    
    for unit, group in df.groupby("unit"):
        # Sort observations by operational flight cycle
        group_sorted = group.sort_values("cycle").reset_index(drop=True)
        features = group_sorted[feature_cols].values
        targets = group_sorted[target_col].values
        cycles = group_sorted["cycle"].values
        
        num_windows = len(group_sorted) - seq_len + 1
        for i in range(num_windows):
            # Extract 30 consecutive observations
            window_features = features[i : i + seq_len]
            # Target is the RUL of the final observation in the window
            window_target = targets[i + seq_len - 1]
            
            X_list.append(window_features)
            y_list.append(window_target)
            metadata_list.append({
                "unit": unit,
                "start_cycle": cycles[i],
                "end_cycle": cycles[i + seq_len - 1],
                "target_rul": window_target
            })
            
    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y, metadata_list

# Generate training windows
X_train, y_train, train_window_meta = extract_sliding_windows(
    train_split_df, WINDOW_SIZE, selected_features, target_col="RUL"
)

print("=" * 65)
print("STAGE 2: TRAINING WINDOWS GENERATED")
print("=" * 65)
print(f"X_train shape : {X_train.shape}  (dtype: {X_train.dtype})")
print(f"y_train shape : {y_train.shape}       (dtype: {y_train.dtype})")
print(f"Total training windows extracted: {len(X_train):,}")
print("=" * 65)

STAGE 2: TRAINING WINDOWS GENERATED
X_train shape : (14241, 30, 17)  (dtype: float32)
y_train shape : (14241,)       (dtype: float32)
Total training windows extracted: 14,241


---
## Stage 3: Generate Validation Windows

The exact same sliding window extraction process is applied to the 20 validation engines.
- **Engine Boundary Integrity:** Windows are constructed strictly within each engine's lifetime; no sequence ever bridges across two different engines.
- **Scale Consistency:** Validation sequences use the same scaled feature representations generated from the training-fitted scaler.

In [4]:
# Generate validation windows
X_val, y_val, val_window_meta = extract_sliding_windows(
    val_split_df, WINDOW_SIZE, selected_features, target_col="RUL"
)

print("=" * 65)
print("STAGE 3: VALIDATION WINDOWS GENERATED")
print("=" * 65)
print(f"X_val shape : {X_val.shape}  (dtype: {X_val.dtype})")
print(f"y_val shape : {y_val.shape}       (dtype: {y_val.dtype})")
print(f"Total validation windows extracted: {len(X_val):,}")
print("=" * 65)

STAGE 3: VALIDATION WINDOWS GENERATED
X_val shape : (3490, 30, 17)  (dtype: float32)
y_val shape : (3490,)       (dtype: float32)
Total validation windows extracted: 3,490


---
## Stage 4: Generate Test Windows

### Benchmark Test Evaluation Protocol
Unlike training engines that are monitored until complete breakdown, test engines represent operational in-service aircraft whose data acquisition was truncated at an undisclosed operational cycle.

- For each test engine, the task is to predict its remaining useful life **at the latest point in time** (the end of its recorded time series).
- We extract **exactly one sequence per test engine** using its **final 30 observed cycles** ($[-30:]$).
- `y_test` is matched from `RUL_FD001.txt`, which provides the scalar true RUL remaining after the final cycle of each test engine.

In [5]:
X_test_list = []
test_engine_ids = []
y_test_list = []
test_window_meta = []

for unit, group in test_df.groupby("unit"):
    group_sorted = group.sort_values("cycle").reset_index(drop=True)
    
    # Confirm the engine has at least 30 observations
    assert len(group_sorted) >= WINDOW_SIZE, f"Test unit {unit} has fewer than {WINDOW_SIZE} cycles!"
    
    # Extract only the final 30 observed cycles
    final_window_features = group_sorted[selected_features].values[-WINDOW_SIZE:]
    start_cycle = group_sorted["cycle"].iloc[-WINDOW_SIZE]
    end_cycle = group_sorted["cycle"].iloc[-1]
    
    # Ground-truth RUL from rul_df (unit is 1-indexed, rul_df is 0-indexed)
    true_rul = float(rul_df.iloc[unit - 1]["RUL"])
    
    X_test_list.append(final_window_features)
    test_engine_ids.append(unit)
    y_test_list.append(true_rul)
    test_window_meta.append({
        "unit": unit,
        "start_cycle": start_cycle,
        "end_cycle": end_cycle,
        "target_rul": true_rul
    })

X_test = np.array(X_test_list, dtype=np.float32)
test_engine_ids = np.array(test_engine_ids, dtype=int)
y_test = np.array(y_test_list, dtype=np.float32)

print("=" * 65)
print("STAGE 4: TEST WINDOWS GENERATED")
print("=" * 65)
print(f"X_test shape          : {X_test.shape}  (dtype: {X_test.dtype})")
print(f"y_test shape          : {y_test.shape}         (dtype: {y_test.dtype})")
print(f"test_engine_ids shape : {test_engine_ids.shape}       (units: {test_engine_ids.min()} to {test_engine_ids.max()})")
print("=" * 65)

STAGE 4: TEST WINDOWS GENERATED
X_test shape          : (100, 30, 17)  (dtype: float32)
y_test shape          : (100,)         (dtype: float32)
test_engine_ids shape : (100,)       (units: 1 to 100)


---
## Stage 5: Validation, Inspection, and Protocol Analysis

We verify eight core integrity constraints:
1. **Window Length Uniformity:** Every window has precisely 30 time steps.
2. **Feature Count Consistency:** All input tensors share the identical 17 feature channels.
3. **No Boundary Crossing:** All windows are contiguous sequences from a single engine.
4. **Test Window Completeness:** Exactly 100 test engines with 1 final window each.
5. **Test ID & Label Alignment:** Test engine IDs correspond 1:1 with ground-truth test labels.
6. **Numerical Cleanliness:** Zero NaN or infinite values exist in any array.
7. **Target Dimensions:** All target vectors are 1D arrays matching sample counts.
8. **Target Alignment:** Window target values strictly equal the RUL at each window's final cycle.

In [6]:
print("=" * 65)
print("STAGE 5: SYSTEMATIC PIPELINE ASSERTIONS")
print("=" * 65)

# 1. Window length checks
assert X_train.shape[1] == 30, "X_train window length != 30"
assert X_val.shape[1] == 30, "X_val window length != 30"
assert X_test.shape[1] == 30, "X_test window length != 30"
print("✓ Check 1 Passed: All windows have exactly 30 time steps.")

# 2. Feature dimension checks
assert X_train.shape[2] == 17, "X_train features != 17"
assert X_val.shape[2] == 17, "X_val features != 17"
assert X_test.shape[2] == 17, "X_test features != 17"
print("✓ Check 2 Passed: All three input arrays have identical feature counts (17).")

# 3. Engine boundary integrity (end_cycle - start_cycle == 29)
train_spans_valid = all(m["end_cycle"] - m["start_cycle"] == 29 for m in train_window_meta)
val_spans_valid = all(m["end_cycle"] - m["start_cycle"] == 29 for m in val_window_meta)
assert train_spans_valid and val_spans_valid, "Window crosses engine boundaries!"
print("✓ Check 3 Passed: Training and validation windows contain no mixed-engine sequences.")

# 4. Test engine count
assert len(X_test) == 100, "Test window count != 100"
assert len(test_engine_ids) == 100, "Test engine ID count != 100"
print("✓ Check 4 Passed: Every test engine contributes exactly one final window (100 total).")

# 5. Test ID alignment
assert list(test_engine_ids) == list(range(1, 101)), "Test engine IDs do not match 1..100!"
assert np.allclose(y_test, rul_df["RUL"].values.astype(np.float32)), "Test RUL values mismatch!"
print("✓ Check 5 Passed: Test windows correspond to correct engine IDs and ground-truth RUL.")

# 6. Absence of NaN / Infs
assert not np.isnan(X_train).any() and not np.isnan(y_train).any(), "NaN in train!"
assert not np.isnan(X_val).any() and not np.isnan(y_val).any(), "NaN in val!"
assert not np.isnan(X_test).any() and not np.isnan(y_test).any(), "NaN in test!"
assert not np.isinf(X_train).any() and not np.isinf(X_val).any() and not np.isinf(X_test).any(), "Inf found!"
print("✓ Check 6 Passed: No NaN or infinite values exist in any array.")

# 7. Target array dimensions
assert y_train.ndim == 1 and len(y_train) == len(X_train), "y_train shape mismatch!"
assert y_val.ndim == 1 and len(y_val) == len(X_val), "y_val shape mismatch!"
assert y_test.ndim == 1 and len(y_test) == len(X_test), "y_test shape mismatch!"
print("✓ Check 7 Passed: All target arrays have expected 1D dimensions matching samples.")

# 8. Target alignment verification with final cycle
train_target_align = all(y_train[i] == train_window_meta[i]["target_rul"] for i in range(len(y_train)))
val_target_align = all(y_val[i] == val_window_meta[i]["target_rul"] for i in range(len(y_val)))
assert train_target_align and val_target_align, "Target alignment mismatch!"
print("✓ Check 8 Passed: Window targets strictly match RUL at the window's final cycle.")
print("=" * 65)

STAGE 5: SYSTEMATIC PIPELINE ASSERTIONS
✓ Check 1 Passed: All windows have exactly 30 time steps.
✓ Check 2 Passed: All three input arrays have identical feature counts (17).
✓ Check 3 Passed: Training and validation windows contain no mixed-engine sequences.
✓ Check 4 Passed: Every test engine contributes exactly one final window (100 total).
✓ Check 5 Passed: Test windows correspond to correct engine IDs and ground-truth RUL.
✓ Check 6 Passed: No NaN or infinite values exist in any array.
✓ Check 7 Passed: All target arrays have expected 1D dimensions matching samples.
✓ Check 8 Passed: Window targets strictly match RUL at the window's final cycle.


In [7]:
print("=" * 65)
print("PREPARED SEQUENCE TENSORS SUMMARY")
print("=" * 65)
print(f"Training Windows     : {len(X_train):,}")
print(f"Validation Windows   : {len(X_val):,}")
print(f"Test Windows         : {len(X_test):,}")
print("-" * 65)
print(f"X_train shape        : {X_train.shape}  | y_train shape: {y_train.shape}")
print(f"X_val shape          : {X_val.shape}   | y_val shape  : {y_val.shape}")
print(f"X_test shape         : {X_test.shape}    | y_test shape : {y_test.shape}")
print(f"Selected Features    : {len(selected_features)} features")
print(f"Window Time Steps    : {WINDOW_SIZE} cycles")
print("=" * 65)

# Display small example tracking details
print("\nEXAMPLE WINDOW DETAILS:")
example_indices = [0, 100, 14240]
example_rows = []
for idx in example_indices:
    m = train_window_meta[idx]
    example_rows.append({
        "Window Index": idx,
        "Engine Unit": m["unit"],
        "Start Cycle": m["start_cycle"],
        "End Cycle": m["end_cycle"],
        "Target RUL": m["target_rul"],
        "Window Shape": str(X_train[idx].shape)
    })
display(pd.DataFrame(example_rows))

PREPARED SEQUENCE TENSORS SUMMARY
Training Windows     : 14,241
Validation Windows   : 3,490
Test Windows         : 100
-----------------------------------------------------------------
X_train shape        : (14241, 30, 17)  | y_train shape: (14241,)
X_val shape          : (3490, 30, 17)   | y_val shape  : (3490,)
X_test shape         : (100, 30, 17)    | y_test shape : (100,)
Selected Features    : 17 features
Window Time Steps    : 30 cycles

EXAMPLE WINDOW DETAILS:


,Window Index,Engine Unit,Start Cycle,End Cycle,Target RUL,Window Shape
0,0,2,1,30,125,"(30, 17)"
1,100,2,101,130,125,"(30, 17)"
2,14240,100,171,200,0,"(30, 17)"


### Why Does the Test Protocol Use Only the Final 30 Observed Cycles?

In operational predictive maintenance, the deployment scenario differs fundamentally between model training and real-time inference:
1. **Operational Maintenance Objective:** An in-service turbofan engine is inspected at its current, latest flight cycle. Fleet operators need to know: *"Given all telemetry up to today, how many flight cycles can this engine safely complete before mandatory overhaul?"*
2. **Evaluation at the Point of Truncation:** The NASA C-MAPSS benchmark intentionally halts each test engine's time series at an arbitrary operational cycle before system failure. The ground-truth RUL value in `RUL_FD001.txt` represents the remaining cycles precisely at this cutoff point.
3. **Standard Benchmark Consistency:** Generating sliding windows throughout earlier test cycles would evaluate intermediate points for which ground-truth labels are not provided by NASA's evaluation protocol. Therefore, using the most recent 30-cycle temporal window ($[-30:]$) directly forecasts the remaining operational horizon at the latest known state.

---
### Summary
- **Training Set:** 14,241 sequences of shape `(30, 17)` across 80 training engines.
- **Validation Set:** 3,490 sequences of shape `(30, 17)` across 20 validation engines.
- **Test Set:** 100 sequences of shape `(30, 17)` corresponding to the final 30 observed cycles of all 100 test engines.
- **Data Types:** All input and target arrays are stored as standard `float32` arrays, ready for direct ingestion by deep learning architectures (MLP, 1D-CNN, LSTM, BiLSTM).
- No models have been trained, evaluated, or saved.